# BIO 2910 · Assignment 07-01 · Multiple Sequence Alignments (in class)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakeSaunders/BIO2910-Bioinformatics/blob/main/Unit07/BIO2910_Assignment_07-01_MSA_inclass.ipynb)

**Today you will:**
1. Learn just enough **Python** to read and work with sequences (Part 0)
2. Use DNA barcodes and a multiple sequence alignment to **catch seafood fraud** (Part 1)
3. Build a **sequence logo** that shows one of the most conserved motifs in biology: the potassium channel selectivity filter (Part 2)

---

### Before you start: save your own copy
This notebook opens **read-only** from GitHub. Go to **File → Save a copy in Drive** *now*, or your work will disappear when you close the tab.

### How Colab works
* A notebook is made of **cells**. Grey boxes are **code cells**; everything else is a **text cell**.
* Click in a code cell and press **Shift + Enter** to run it. The output appears below the cell.
* Run cells **in order, top to bottom**. If something breaks, try **Runtime → Restart session and run all**.
* Nothing to install on your laptop. Everything runs on a Google server.

---
# Part 0 · A 20-minute Python crash course (for R users)

You already know R, and Python works very similarly. The biggest differences:

| Idea | R | Python |
|---|---|---|
| assign a variable | `x <- 5` | `x = 5` |
| text (a "string") | `"ATG"` | `"ATG"` |
| a vector / list | `c(1, 2, 3)` | `[1, 2, 3]` |
| a named lookup | `c(A = 1.8, R = -4.5)` | `{"A": 1.8, "R": -4.5}` (a **dictionary**) |
| first element | `x[1]` | `x[0]`  ← **Python counts from 0!** |
| length | `length(x)` / `nchar(x)` | `len(x)` |
| print | `print(x)` | `print(x)` |
| comments | `# comment` | `# comment` |
| loops and functions | `{ }` braces | a colon `:` and **indentation** (4 spaces) |

### 0.1 Strings: sequences are just text
Run the cell below. Then change the sequence and run it again.

In [ ]:
dna = "ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG"

print(dna)
print("Length:", len(dna))

# Indexing starts at 0 in Python. dna[0] is the FIRST base.
print("First base:", dna[0])

# Slicing: dna[start:stop] goes from start up to (but NOT including) stop
print("First codon:", dna[0:3])
print("Last three bases:", dna[-3:])

# Strings have built-in "methods" you call with a dot
print("Number of G:", dna.count("G"))
print("Lower case:", dna.lower())

### 0.2 Lists and `for` loops
A list holds many items in order. A `for` loop runs the **indented** code once for every item.

In [ ]:
codons = []                           # an empty list
for i in range(0, len(dna), 3):       # i = 0, 3, 6, 9, ...
    codons.append(dna[i:i+3])         # add one codon to the end of the list

print(codons)
print("Number of codons:", len(codons))
print("Is there a stop codon (TAG)?", "TAG" in codons)

### 0.3 Dictionaries: look things up by name
A dictionary maps **keys** to **values**. It is perfect for things like the complement of each base.

In [ ]:
complement = {"A": "T", "T": "A", "G": "C", "C": "G"}

print(complement["G"])     # look up one key

# Build the reverse complement one base at a time
rev_comp = ""
for base in reversed(dna):
    rev_comp = rev_comp + complement[base]

print(dna)
print(rev_comp)

### 0.4 Functions: package code so you can reuse it
`def` creates a function. Whatever comes after `return` is the answer the function gives back.

In [ ]:
def gc_content(seq):
    """Return the percent of bases in seq that are G or C."""
    gc = seq.count("G") + seq.count("C")
    return 100 * gc / len(seq)

print(gc_content(dna))
print(round(gc_content("GGGGCCAT"), 1))

# f-strings let you drop variables straight into text. Note the f before the quote:
print(f"This sequence is {len(dna)} bp long and {gc_content(dna):.1f}% GC")

> **✏️ Try it (Question 1):** Finish the `at_content()` function below so it returns the percent of bases that are A or T.
> Replace each `___` and run the cell. What do `gc_content(dna)` and `at_content(dna)` add up to? Why?
>
> *Hint:* `at_content()` is almost a copy of `gc_content()` above. Change which two letters you count, and use your new variable in the `return` line.

In [ ]:
# Question 1: replace each ___ , then run the cell

def at_content(seq):
    """Return the percent of bases in seq that are A or T."""
    at = seq.count("___") + seq.count("___")   # 1) count the A's and the T's
    return 100 * ___ / len(seq)                 # 2) turn the count into a percent

print("AT content:", at_content(dna))
print("GC + AT =", gc_content(dna) + at_content(dna))

---
# Setup · install and load the MSA tools

R has packages, and Python has them too. Colab already includes a lot of them, so we only need three extras:

* **pyFAMSA** builds the alignments with FAMSA, an aligner fast enough to align thousands of sequences on a laptop
* **pyMSAviz** draws the alignments
* **Logomaker** draws sequence logos

The `!` at the start tells Colab to run the line as a terminal command, not Python. This takes about 10 seconds.

In [ ]:
!pip install -q pyfamsa pymsaviz logomaker

In [ ]:
# import is Python's version of library()
import urllib.request                 # download files from the web (built into Python)
import pandas as pd                   # data frames (like R's data.frame / tibble)
import matplotlib.pyplot as plt       # plotting (like base R plots / ggplot2)
import seaborn as sns                 # nicer statistical plots, built on matplotlib
import logomaker                      # sequence logos
from pyfamsa import Aligner, Sequence # the aligner
from pymsaviz import MsaViz           # the alignment viewer

# All the data for this unit lives in the course GitHub repository
BASE = "https://raw.githubusercontent.com/JakeSaunders/BIO2910-Bioinformatics/main/data/"
print("Ready!")

### Three helper functions we'll use all unit

You don't need to memorize these, but read the comments. Every line uses something from Part 0.

* `read_fasta()` turns a FASTA file into a **dictionary**: `{name: sequence}`
* `align()` runs FAMSA on that dictionary and returns an aligned dictionary (same names, but now with `-` gaps)
* `write_fasta()` saves a dictionary to a FASTA file (pyMSAviz plots from a file)

In [ ]:
def read_fasta(url):
    """Read a FASTA file from a URL. Return a dictionary of {name: sequence}."""
    text = urllib.request.urlopen(url).read().decode()
    seqs = {}
    for line in text.splitlines():
        line = line.strip()
        if line.startswith(">"):                 # a header line starts a new record
            name = line[1:].split()[0]           # keep only the first word as the name
            seqs[name] = ""
        elif line:                               # sequence lines get glued onto that record
            seqs[name] += line.upper()
    return seqs


def align(seqs):
    """Align a {name: sequence} dictionary with FAMSA. Return {name: aligned sequence}."""
    aligner = Aligner(guide_tree="upgma")
    msa = aligner.align([Sequence(n.encode(), s.encode()) for n, s in seqs.items()])
    aligned = {m.id.decode(): m.sequence.decode() for m in msa}
    return {n: aligned[n] for n in seqs}        # keep the original order


def write_fasta(seqs, filename):
    """Save a {name: sequence} dictionary as a FASTA file."""
    with open(filename, "w") as f:
        for name, s in seqs.items():
            f.write(f">{name}\n{s}\n")
    print("Saved", filename)

---
# Part 1 · Seafood fraud: what's really on your plate?

When fish is filleted, cooked or sliced for sushi, you can't tell species apart by looking at them. That makes seafood one of the most commonly **mislabeled** foods. A cheap species gets sold as an expensive one, a farmed fish gets sold as wild, and sometimes the substitute is a health risk. *Escolar*, sold as "white tuna," contains indigestible wax esters that cause severe digestive upset.

**DNA barcoding** solves this. A ~650 bp piece of the mitochondrial gene **cytochrome c oxidase subunit I (COI)** varies little *within* a species but differs *between* species. If you sequence the COI barcode from a mystery sample and compare it to reference barcodes from known species, the closest match is (usually) the species you ate.

**Your data** (all real sequences from GenBank):
* `seafood_COI_references.fasta`: reference barcodes for 14 species commonly sold in the U.S.
* `seafood_COI_samples.fasta`: barcodes from **7 "restaurant and market samples"** labelled with what they were *sold as*

Your job: **figure out what each sample really is, and decide which ones were mislabeled.**

In [ ]:
refs = read_fasta(BASE + "seafood_COI_references.fasta")
samples = read_fasta(BASE + "seafood_COI_samples.fasta")

print(f"{len(refs)} reference sequences:")
for name, s in refs.items():
    print(f"  {name:16s} {len(s)} bp")

print(f"\n{len(samples)} mystery samples:")
for name, s in samples.items():
    print(f"  {name:16s} {len(s)} bp")

`read_fasta()` keeps only the first word of each header as the name. To see the scientific names and what each sample was **sold as**, print the full headers:

In [ ]:
for f in ["seafood_COI_references.fasta", "seafood_COI_samples.fasta"]:
    text = urllib.request.urlopen(BASE + f).read().decode()
    for line in text.splitlines():
        if line.startswith(">"):
            print(line)
    print()

### 1.1 Align all 21 sequences
Notice that the sequences are **different lengths**. Each lab trimmed its reads a little differently. An alignment inserts gaps (`-`) so that the same position in the gene lines up in the same column across every sequence.

In R, this step took minutes with the `msa` package. Watch how long FAMSA takes:

In [ ]:
%%time
all_seqs = {**refs, **samples}          # ** merges two dictionaries into one
fish_aln = align(all_seqs)

aln_length = len(fish_aln["Sample_A"])
print("Alignment length:", aln_length, "columns")
write_fasta(fish_aln, "seafood_COI_alignment.fasta")

### 1.2 Look at the alignment
`MsaViz` draws an alignment like `ggmsa` did in R. `start` and `end` choose which columns to show.

In [ ]:
mv = MsaViz("seafood_COI_alignment.fasta", color_scheme="Nucleotide",
            start=1, end=120, wrap_length=60, show_consensus=True, show_count=True)
fig = mv.plotfig()
fig.savefig("seafood_alignment_1-120.png", dpi=150, bbox_inches="tight")

> **✏️ Question 2:** Look at the first ~40 columns. Why do some sequences start with a run of gaps (`-`)? Is that a biological difference between the fish, or something else?
>
> **✏️ Question 3:** Find a few columns where the 5 tunas (*Thunnus* spp. and *Katsuwonus pelamis*) differ from each other. Are the differences scattered or clustered? Change `start` and `end` above to explore.

### 1.3 Percent identity: put a number on similarity
For every pair of sequences, we'll count the columns where **both** have a base (no gap) and ask what percent of those are **identical**. We skip gaps so that a sequence that is just *shorter* isn't penalized.

In [ ]:
def percent_identity(a, b):
    """Percent identity of two ALIGNED sequences, ignoring columns with a gap in either."""
    compared = 0
    same = 0
    for x, y in zip(a, b):              # zip() walks down both sequences together
        if x != "-" and y != "-":
            compared += 1
            if x == y:
                same += 1
    return 100 * same / compared


# quick test
print(percent_identity("ACGT-A", "ACCTTA"))   # 4 of 5 compared columns match -> 80.0

In [ ]:
# Compare every sample against every reference (rows = samples, columns = references)
pid = pd.DataFrame(
    [[percent_identity(fish_aln[s], fish_aln[r]) for r in refs] for s in samples],
    index=list(samples), columns=list(refs))

plt.figure(figsize=(12, 4.5))
sns.heatmap(pid, annot=True, fmt=".0f", cmap="viridis", vmin=75, vmax=100,
            cbar_kws={"label": "% identity"})
plt.title("Mystery samples vs. reference COI barcodes")
plt.tight_layout()
plt.savefig("seafood_identity_heatmap.png", dpi=150)
plt.show()

### 1.4 Who's the best match?
For each sample, find the top two reference matches. The **gap** between the best and second-best match tells you how confident the identification is.

In [ ]:
rows = []
for s in samples:
    ranked = pid.loc[s].sort_values(ascending=False)
    rows.append({"sample": s,
                 "best match": ranked.index[0], "best %": round(ranked.iloc[0], 1),
                 "2nd match": ranked.index[1], "2nd %": round(ranked.iloc[1], 1),
                 "gap": round(ranked.iloc[0] - ranked.iloc[1], 1)})
best = pd.DataFrame(rows)
best

> **✏️ Question 4:** Fill in a table with one row per sample: *sold as*, *best match (common name)*, *% identity*, and your verdict: **honest** or **mislabeled**. (Reference names are common names; the full headers also give the scientific name and GenBank accession.)
>
> **✏️ Question 5:** For most samples the best match is ≥ 97% and the second match is well below it. Which sample has a **tiny gap** between its top two matches? Can you be confident in that identification from COI alone? What would you do next to settle it?
>
> **✏️ Question 6:** Look at the full heatmap. Which *references* are most similar to each other? Does that match what you'd expect from their names (genus)?
>
> **✏️ Question 7:** Pick the sample you think was mislabeled in the most harmful way. Explain who is harmed: the customer's wallet, their health, or the wild fish population?

---
# Part 2 · Sequence logos: the potassium channel selectivity filter

Every time a neuron fires, **K⁺ channels** let potassium out to reset the membrane potential. They are astonishingly selective: they let K⁺ through ~1,000 times more easily than Na⁺, even though Na⁺ is *smaller*. The trick is a short loop of the protein called the **selectivity filter**. Its backbone oxygens line the pore at exactly the right spacing to strip water off a K⁺ ion, but not off a Na⁺ ion.

Rod MacKinnon won the 2003 Nobel Prize in Chemistry for solving the structure of the bacterial K⁺ channel **KcsA** and showing how this filter works.

**Your data:** the pore region (pore helix + selectivity filter + start of the inner helix) from **21 channels** in `K_channel_pore.fasta`:
* bacteria and archaea (KcsA, MthK, KvAP)
* the fruit fly *Shaker* channel
* human voltage-gated (Kv), inward-rectifier (Kir), calcium-activated (BK, SK) and cardiac (hERG, KCNQ1) channels
* one mystery channel from the bacterium *Bacillus cereus* (`Mystery_Bacillus`)

If the same short motif shows up in a bacterium and in your heart, it has been conserved for billions of years.

In [ ]:
kchan = read_fasta(BASE + "K_channel_pore.fasta")
for name, s in kchan.items():
    print(f"{name:18s} {s}")

In [ ]:
k_aln = align(kchan)
write_fasta(k_aln, "K_channel_alignment.fasta")

mv = MsaViz("K_channel_alignment.fasta", color_scheme="Zappo", show_consensus=True)
fig = mv.plotfig()
fig.savefig("K_channel_alignment.png", dpi=150, bbox_inches="tight")

### 2.1 Build the logo
A **sequence logo** summarizes an alignment column by column:
* the **height of the whole stack** = how conserved that column is (information, in bits; max ≈ 4.3 for proteins)
* the **height of each letter** = how often that amino acid appears in the column

Logomaker needs a matrix of counts per column, and it can make one directly from a list of aligned sequences.

In [ ]:
info = logomaker.alignment_to_matrix(list(k_aln.values()), to_type="information", pseudocount=0,
                                     characters_to_ignore="-")
fig, ax = plt.subplots(figsize=(14, 3))
logo = logomaker.Logo(info, ax=ax, color_scheme="chemistry")
ax.set_xlabel("Alignment column (0-based)")
ax.set_ylabel("Information (bits)")
ax.set_title("K+ channel pore region: 21 channels from bacteria to humans")
plt.tight_layout()
plt.savefig("K_channel_logo.png", dpi=150)
plt.show()

### 2.2 Find the filter and highlight it
Python strings have a `.find()` method, like a tiny BLAST. We'll find the signature motif in KcsA and shade those columns of the logo.

In [ ]:
kcsa = k_aln["KcsA_bacterium"]
start = kcsa.find("TVGYG")
print("TVGYG starts at alignment column", start)

fig, ax = plt.subplots(figsize=(14, 3))
logo = logomaker.Logo(info, ax=ax, color_scheme="chemistry")
logo.highlight_position_range(pmin=start, pmax=start + 4, color="gold", alpha=0.4)
ax.set_ylabel("Information (bits)")
ax.set_title("Selectivity filter highlighted")
plt.tight_layout()
plt.savefig("K_channel_logo_filter.png", dpi=150)
plt.show()

> **✏️ Question 8:** Which five-residue motif forms the selectivity filter? Which positions in it are the most conserved (tallest)? Which position tolerates the most variation, and which amino acids appear there?
>
> **✏️ Question 9:** The cell below prints the 5-residue filter of every channel. Fill in the slice (replace the two `___`) and run it. One channel doesn't end its filter in GYG or GFG. Which one, and what does it have instead?
>
> *Hint:* `start` is the column where the filter begins, and the filter is 5 residues long. In Part 0, `dna[0:3]` took the first 3 bases, so the slice you need is `seq[start : start + ?]`.

In [ ]:
# Question 9: replace the two ___ , then run the cell

for name, seq in k_aln.items():        # go through the channels one at a time
    filter_seq = seq[___ : ___]         # slice out the 5 filter residues
    print(f"{name:18s} {filter_seq}")   # :18s pads the name so the filters line up

> **✏️ Question 10:** That odd channel, `Mystery_Bacillus`, is called **NaK**. It conducts **both Na⁺ and K⁺** about equally well. Researchers have mutated its filter to the K⁺-channel sequence, and the mutant became K⁺-selective. What does that tell you about why the filter is so conserved in every other channel here?

---
### What to turn in
Answer **Questions 1–10** in a Word document. Include these figures (Colab saves them in the **Files** panel 📁 on the left; right-click → Download):
* `seafood_identity_heatmap.png`
* `K_channel_logo_filter.png`

*Python skills used today:* strings, slicing, lists, `for` loops, dictionaries, functions, f-strings and `import`. You'll use all of them again in the homework.